# 반도체 때문에 전체 결과가 달라졌을까?

여기에는 별개의 두 질문이 있다. 먼저 반도체 한 업종이 40개 업종의 추정치에 미친 영향을 본다. 이어서 반도체만 놓고 그 업종의 상환능력이 왜 움직였는지를 재무 항목으로 나눠 본다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 코드의 E는 사전 단기차입 의존도, L은 사전 유동비율이다. Post는 2022~2023년이며 EP는 E와 Post를 곱한 항이다. EP의 계수는 사전 단기차입 의존도가 1%포인트 높은 업종에서 긴축기의 상대적 ICR 변화가 몇 배 달랐는지 나타낸다.


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 반도체 추가 검토
2026-11-03 사용자 승인: 반도체에 대해 추가로 할 수 있는 분석 수행. 기존 주 분석/자료 보존. 새 결과 확인 전 아래 범위 고정. 모든 분석은 기존 결과를 본 뒤의 탐색적 보조분석.

1. 전체240행 M2의 반도체 포함/제외(234행39업종), 경기표본225행 M2·생산통제·출하통제 각각 반도체 제외(219행37업종). 기간, E/L사전값,변환,통제는 유지. 누락 자료 보간 없음. 제외는 원자료 오류 판단이 아닌 영향도 점검.
2. 동일225행에서 생산(출하)증가율 공통항에 반도체더미×해당증가율을 추가해 반도체만 다른 기울기를 허용. 반도체더미 자체는 업종FE와 중복하므로 추가하지 않음. 상호작용 계수는6개 반도체 관측에 의존하므로 그 p/CI를 확증 근거로 사용하지 않고 점추정만 공개. 핵심은 전체 E×Post 추정 민감도. 유의항 선택으로 모형을 교체하지 않음.
3. 반도체 연도별 생산·출하·이익률·이자부담·ICR을 전체 다른39업종 중앙값과 비교. 금액은 단위십억원. 반도체의 매출·영업이익·이자비용을 따로 제시. 산업 중앙값은 기업 가중평균이 아님.
4. pre18–20 대 post22–23의 공시ICR 평균과 평균금액 기반ICR을 구별. 대칭(두 순서 평균) 회계분해를 원금액으로 재계산하고 기존분해와 일치 검증. 2018→19,19→20,20→21,21→22,22→23 연간분해도 전부 공개. 시점을 유의성에 맞춰 선택하지 않음. 40업종 내 변화순위 및 전체평균/중앙값의 반도체 제외 민감도. 금리 인과효과 아님.

추론: 모든 공통계수 CR1+t(G−1). 새로운6모형의 EP에 WCR-S31 귀무제약Rademacher9999회 seed20261103; Holm6 공개(이번6개만). 반도체전용항 추론은 미보고. 새 독립 시계열 회귀·합성통제·다수 시차 탐색은6년자료와 현재 목적에 비추어 실행하지 않음.

근거: Conley & Taber(2011), Review of Economics and Statistics93(1),113–125, 서론113–114쪽 관련 본문 확인(전문 정독 아님). p113: “the noise W does not vanish as the total number of groups grows larger”. 하나의 처치집단 문제를 통해 전체 집단 수만 많다고 특정집단 추론이 해결되지 않음을 설명. 우리 연속형 반도체기울기의 정확한 정리나 해당 논문의 추론법을 적용했다는 뜻은 아님. 단일업종항 p를 확증으로 쓰지 않는 설계상의 주의로 사용. https://users.ssc.wisc.edu/~ctaber/Papers/ctabdd.pdf

재현: requirements.txt 설치, 이 노트북 폴더에서 전체셀 실행. 입력파일 동결및해시 포함. 최초재무자료 구축01,지수파싱09노트북 참조. 기존주모형 변경없음.


In [1]:
# 반도체와 전체 비교 표본 준비
from pathlib import Path
import pandas as pd,numpy as np,json,hashlib
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from wildboottest.wildboottest import wildboottest
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
(BASE / 'figures').mkdir(exist_ok=True)
def read(name):
    return pd.read_csv(BASE / 'inputs' / name)
def save(d, name):
    d.to_csv(BASE / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
for m in json.loads((BASE / 'inputs/manifest.json').read_text(encoding='utf-8')):
    assert hashlib.sha256((BASE / 'inputs' / m['file']).read_bytes()).hexdigest() == m['sha256']
full = read('full_panel.csv')
c = read('cycle_panel.csv')
f = read('finance.csv')
idx = read('indices.csv')
assert len(full)==240 and len(c)==225
semi = 'I3102'
c['semi_production'] = (c.code == semi) * c.production_growth
c['semi_shipment'] = (c.code == semi) * c.shipment_growth
assert (c.code==semi).sum()==6


### 이 셀에서 준비한 자료

이 셀은 숫자를 인쇄하지 않는다. 전체 240행, 경기자료가 있는 225행, 그중 반도체 6행을 확인해 이후 비교의 기준을 맞춘다.

In [2]:
# 공통 추정식 정의
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

# 전체와 반도체 제외 비교
terms=['EP','LP','ET','LT']
fit('F_full',full,terms)
fit('F_no_semi',full[full.code!=semi].reset_index(drop=True),terms)

# 경기표본에서도 반도체 제외
for name,extra in [('M2',[]),('production',['production_growth']),('shipment',['shipment_growth'])]:
 fit('C_'+name,c,terms+extra)
 fit('X_'+name,c[c.code!=semi].reset_index(drop=True),terms+extra)

# 반도체 전용 경기 기울기 추가
for name in ['production', 'shipment']:
    fit('H_' + name, c, terms + [name + '_growth', 'semi_' + name])
a=pd.DataFrame(rows)
# One-industry interaction slopes are descriptive only, not cluster-robust significance claims.
mask = a.term.str.startswith('semi_')
a.loc[mask, ['se', 'lo', 'hi', 'p']] = np.nan
save(a, 'coefficients')
print(a[a.term == 'EP'].to_string(index=False))

# 업종별 경기 기울기 정리
slopes=[]
for name in ['production', 'shipment']:
    r = fits['H_' + name][3]
    slopes.append(dict(indicator=name, others_slope=r.params[name + '_growth'], semi_difference=r.params['semi_' + name], semi_slope=r.params[name + '_growth'] + r.params['semi_' + name], semi_N=6, inference='descriptive only'))
save(pd.DataFrame(slopes),'industry_slopes')

# 제외 모형 재검정
loo = read('prior_loo.csv')
np.testing.assert_allclose(fits['F_no_semi'][3].params.EP, loo.loc[loo.excluded_code == semi, 'EP'].iloc[0], atol=1e-09)
new = ['F_no_semi', 'X_M2', 'X_production', 'X_shipment', 'H_production', 'H_shipment']
boot = []
for label in new:
    dd, tr, model, res = fits[label]
    gg = pd.Series(dd.code.to_numpy())
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    yy = pd.Series(model.endog)
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    np.testing.assert_allclose(mm.fit().params.EP, res.params.EP, atol=1e-08)
    r = wildboottest(mm, param='EP', cluster=pd.factorize(gg)[0], B=9999, seed=20261103, bootstrap_type='31', impose_null=True, weights_type='rademacher', show=False)
    boot.append(dict(model=label, term='EP', p_boot=float(r['p-value'].iloc[0])))
z = a.merge(pd.DataFrame(boot), on=['model', 'term'], validate='one_to_one')
z['p_holm6'] = multipletests(z.p, method='holm')[1]
z['p_boot_holm6'] = multipletests(z.p_boot, method='holm')[1]
save(z, 'inference')
print(z.to_string(index=False))


       model outcome term     beta       se        lo       hi        p   N  G
      F_full       Y   EP 0.545912 0.451865 -0.368072 1.459896 0.234272 240 40
   F_no_semi       Y   EP 0.204812 0.331713 -0.466705 0.876330 0.540628 234 39
        C_M2       Y   EP 0.617624 0.467445 -0.329508 1.564757 0.194521 225 38
        X_M2       Y   EP 0.270099 0.341674 -0.422848 0.963046 0.434402 219 37
C_production       Y   EP 0.606350 0.453572 -0.312675 1.525374 0.189440 225 38
X_production       Y   EP 0.269610 0.341536 -0.423056 0.962277 0.435043 219 37
  C_shipment       Y   EP 0.612669 0.459050 -0.317455 1.542794 0.190148 225 38
  X_shipment       Y   EP 0.272420 0.342956 -0.423127 0.967968 0.432209 219 37
H_production       Y   EP 0.341123 0.343955 -0.355795 1.038042 0.327753 225 38
  H_shipment       Y   EP 0.461888 0.389321 -0.326951 1.250727 0.243029 225 38
       model outcome term     beta       se        lo       hi        p   N  G   p_boot  p_holm6  p_boot_holm6
   F_no_semi       Y

### 출력에서 읽을 수 있는 것

F_full과 F_no_semi를 비교하면 전체 EP는 +0.546→+0.205다. H_production과 H_shipment는 반도체를 포함하고 경기와의 관계만 다르게 허용한 별도 비교다.

- 업종 및 연도 차이를 고려하는 공통 회귀식을 정의했다.
- 전체 M2에서 +0.546이던 단기차입 계수가 반도체 제외 후 +0.205(p=0.541)로 작아졌다.
- 경기자료가 있는 표본에서도 반도체 제외 전 +0.618, 제외 후 +0.270이었다. 반도체를 뺀 뒤에도 예상한 음의 관계가 드러나지 않았다.
- 반도체만 경기지표와 다른 관계를 갖도록 허용하면 전체 단기차입 계수는 생산 모형 +0.341(p=0.328), 출하 모형 +0.462(p=0.243)였다.
- 반도체 전용 경기 기울기는 한 업종의 6개 연도로 계산했다. 다른 업종과의 기울기 차이는 기술적으로만 읽는다.
- 반도체 제외 후 전체 M2의 단기차입 항은 부트스트랩에서도 p=0.548이었다. 검정 방식을 바꿔도 악화 관계가 확인되지 않았다.

In [3]:
# 기간평균 ICR 변화 분해

# Symmetric accounting identity for R=A/B; B>0, negative profits allowed.
def decompose(a0, a1, b0, b1):
    assert b0 > 0 and b1 > 0
    cp = 0.5 * (a1 - a0) * (1 / b0 + 1 / b1)
    ci = 0.5 * (a0 + a1) * (1 / b1 - 1 / b0)
    change = a1 / b1 - a0 / b0
    np.testing.assert_allclose(cp + ci, change, atol=1e-10)
    return (cp, ci, change)
out=[]
for code, g in f.groupby('code'):
    pre = g[g.year.between(2018, 2020)]
    post = g[g.year.between(2022, 2023)]
    A0 = pre['영업이익'].mean()
    A1 = post['영업이익'].mean()
    B0 = pre['이자비용'].mean()
    B1 = post['이자비용'].mean()
    cp, ci, ch = decompose(A0, A1, B0, B1)
    out.append(dict(code=code, industry=g.industry.iloc[0], profit_pre=A0, profit_post=A1, interest_pre=B0, interest_post=B1, ratio_pre=A0 / B0, ratio_post=A1 / B1, profit_contribution=cp, interest_contribution=ci, ratio_change=ch, official_pre=pre.icr.mean(), official_post=post.icr.mean(), official_change=post.icr.mean() - pre.icr.mean(), cost_change=post.cost.mean() - pre.cost.mean(), margin_change=post.margin.mean() - pre.margin.mean()))
dec = pd.DataFrame(out)
dec['decline_rank'] = dec.official_change.rank(method='min', ascending=True).astype(int)
prior = read('prior_decomposition.csv').set_index('code')
test = dec.set_index('code')
np.testing.assert_allclose(test.loc[prior.index,'ratio_change'],prior.reconstructed_change,atol=1e-9)
save(dec, 'period_decomposition_all40')
print('Semiconductor period decomposition:')
print(dec[dec.code == semi].to_string(index=False))

# 반도체 연간 변화 분해
sg = f[f.code == semi].sort_values('year')
annual = []
for j in range(1, len(sg)):
    x, y = (sg.iloc[j - 1], sg.iloc[j])
    cp, ci, ch = decompose(x['영업이익'], y['영업이익'], x['이자비용'], y['이자비용'])
    annual.append(dict(start=int(x.year), end=int(y.year), profit_contribution=cp, interest_contribution=ci, amount_ratio_change=ch, official_icr_change=y.icr - x.icr))
save(pd.DataFrame(annual),'semi_annual_decomposition')

# 전체 평균과 중앙값 비교
merged=f.merge(idx[['code','year','production','shipment','production_growth','shipment_growth']],on=['code','year'],how='left',validate='one_to_one')
series = merged[merged.code == semi].copy()
save(series, 'semi_annual_finance_indices')
other = merged[merged.code != semi].groupby('year')[['icr', 'cost', 'margin', 'burden', 'production_growth', 'shipment_growth']].agg(['median', 'count'])
other.columns = ['_'.join(x) for x in other.columns]
save(other.reset_index(), 'other_industries_annual')
summ=[]
for name,g in [('all40',dec),('without_semi39',dec[dec.code!=semi])]:
 summ.append(dict(sample=name,N=len(g),mean_icr_change=g.official_change.mean(),median_icr_change=g.official_change.median(),icr_decline_count=int((g.official_change<0).sum())))
save(pd.DataFrame(summ), 'aggregate_sensitivity')
print(pd.DataFrame(summ).to_string(index=False))

# 연간 흐름과 기여분 그리기
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
fig,axs=plt.subplots(2,2,figsize=(12,8))
for ax, (v, title) in zip(axs.flat, [('icr', 'ICR (배)'), ('cost', '비용 대용률 (%)'), ('margin', '영업이익률 (%)'), ('production_growth', '생산 로그증가율 (×100)')]):
    ax.plot(series.year, series[v], marker='o', label='반도체')
    ax.plot(other.index, other[v + '_median'], marker='s', label='다른 업종 중앙값')
    ax.axvspan(2021.75, 2023.15, color='gray', alpha=0.12)
    ax.set_title(title)
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle('반도체와 다른 업종의 연도별 변화 — 음영: 2022~2023')
fig.tight_layout()
fig.savefig(BASE / 'figures/semi_trends.png', dpi=150)
plt.close(fig)
ad = pd.DataFrame(annual)
fig, ax = plt.subplots(figsize=(10, 5))
xx = np.arange(len(ad))
ax.bar(xx - 0.18, ad.profit_contribution, 0.36, label='영업이익 기여')
ax.bar(xx + 0.18, ad.interest_contribution, 0.36, label='이자비용 기여')
ax.set_xticks(xx, [str(x) + '→' + str(y) for x, y in zip(ad.start, ad.end)])
ax.axhline(0, color='gray')
ax.set_ylabel('금액 기반 ICR 변화 (배)')
ax.set_title('반도체 연간 ICR 변화의 회계적 분해')
ax.legend()
fig.tight_layout()
fig.savefig(BASE / 'figures/semi_decomposition.png', dpi=150)
plt.close(fig)


Semiconductor period decomposition:
 code industry   profit_pre  profit_post  interest_pre  interest_post  ratio_pre  ratio_post  profit_contribution  interest_contribution  ratio_change  official_pre  official_post  official_change  cost_change  margin_change  decline_rank
I3102      반도체 10284.333333       3869.0    350.666667         1130.0  29.327947    3.423894           -11.985981             -13.918071    -25.904053     33.794667         7.3605       -26.434167     0.589696     -14.241564             2
        sample  N  mean_icr_change  median_icr_change  icr_decline_count
         all40 40        -0.871550             -0.644                 24
without_semi39 39        -0.216098             -0.636                 23


### 출력에서 읽을 수 있는 것

반도체 제외 전후 업종 평균 ICR 변화는 −0.872→−0.216배, 중앙값은 −0.644→−0.636배였다. 반도체 포함 여부에 평균이 더 민감했다.

- 기간평균 금액으로 재구성한 ICR 변화를 영업이익과 이자비용의 기여로 분리해 기존 분해표와 대조했다.
- 반도체의 2022→2023년 재구성 ICR 하락은 약 19.13배였다. 이 중 영업이익 변화 기여는 −15.74배, 이자비용 변화 기여는 −3.39배로 이 해에는 영업이익 쪽이 더 컸다.
- 전체 40개 업종의 공시 ICR 기간 변화 평균은 −0.872배, 반도체를 뺀 39개 업종은 −0.216배였다. 중앙값은 각각 −0.644배와 −0.636배로 비슷했다. 반도체 제외가 평균과 중앙값에 미치는 영향이 달랐다.
- 연도별 그림에서는 반도체의 경로와 다른 업종 요약치를, 분해 그림에서는 같은 해의 이익 기여와 이자 기여를 비교한다.

![반도체와 다른 업종의 연도별 흐름](figures/semi_trends.png)

*반도체와 다른 업종의 연도별 흐름*

![반도체 ICR 변화의 연도별 분해](figures/semi_decomposition.png)

*반도체 ICR 변화의 연도별 분해*

In [4]:
# 사전 단기차입 의존도 순위 확인
from pathlib import Path
import pandas as pd
df=pd.read_csv(Path.cwd()/'inputs/full_panel.csv')[['code','industry','E','L']].drop_duplicates()
df['E_low_rank']=df.E.rank(method='min')
print(df[df.code == 'I3102'].to_string(index=False))
print('E median:', df.E.median())
df.to_csv(Path.cwd()/'tables/pre_exposure_ranks.csv',index=False,encoding='utf-8-sig')


 code industry        E          L  E_low_rank
I3102      반도체 2.115468 154.912003         1.0
E median: 9.66050527517083


### 출력에서 읽을 수 있는 것

E=2.115%는 반도체의 사전 단기차입 의존도다. 전체 업종 중앙값은 9.661%이고 낮은 순위는 1위였다. L=154.912%는 별도 변수인 사전 유동비율이다.

# 반도체 추가분석 요약
전체M2 EP .5459→반도체제외 .2048(p .5406, bootstrap .5479).
경기표본 M2 EP .6176→제외 .2701(p .4344). 제외후 생산 .2696,출하 .2724.
반도체 별도 생산/출하기울기 허용시 핵심EP .3411(p .3278)/.4619(p .2430). 반도체 전용항은 단일업종6관측으로 점추정만 공개.
공시ICR 기간평균33.7947→7.3605, 변화−26.4342(40업종 하락2위).
평균금액비율29.3279→3.4239, 변화−25.9041=영업이익−11.9860+이자비용−13.9181.
2021→22 이자기여 절댓값이 더 크고2022→23 이익기여가 더 큼. 금리 인과효과 아님.
전체 설명과 기존 분석 통합은 ../integrated_report_v3/팀공유_전체분석_통합보고서_v3.html 참조.
